In [1]:
import pandas as pd
import os

In [2]:
tracker_rel_path = '/General/Market Research/Trackers - EUROPE/Prices/Tracker/Exus power price forecasts tracker.xlsx'

In [3]:
sp_path = os.environ['SP_BASE_PATH']
sp_path

'C:\\Users\\mpy\\OneDrive - Exus Management Partners\\EU - Strategy & Markets - Documentos'

In [4]:
tracker_abs_path = os.path.join(sp_path, tracker_rel_path.lstrip('/'))

df_raw = pd.read_excel(tracker_abs_path, sheet_name='Real prices', skiprows=5, usecols='C:BH')

df_raw = df_raw[df_raw['Source'] == 'Aurora']

In [5]:
from common_libs.utils.utils_dates import map_month_abbr

df_raw['Month'] = df_raw['Release Month/quarter'].apply(map_month_abbr)

df_raw['Date'] = pd.to_datetime(df_raw['Release year'].astype(int).astype(str) + '-' + df_raw['Month'].astype(str), format='%Y-%m')

In [6]:
df_raw.sort_values(by=['Country', 'Scenario', 'Curtailed/uncurtailed', 'Scope', 'Technology view', 'Date'], inplace=True)

df_raw_fil = df_raw[
    (df_raw['Technology view'].isin(['Solar PV', 'Onshore wind']) | df_raw['Technology view'].isna()) & 
    (df_raw['Scenario'] == 'Central') &
    ((df_raw['Curtailed/uncurtailed'] == 'Uncurtailed') | df_raw['Curtailed/uncurtailed'].isna())
    ].reset_index(drop=True)
type(df_raw_fil)
df_raw_fil

,Country,Source,Release year,Release Month/quarter,Release date,Scenario,Curtailed/uncurtailed,Scope,Technology view,Year of the real currency,...,2055,2056,2057,2058,2059,2060,2061,2062,Month,Date
0,France,Aurora,2024,Jan,2024-Jan,Central,Uncurtailed,Capture,Onshore wind,2022,...,70.970334,70.970334,70.530889,70.970334,71.629501,69.542138,NaN,NaN,1,2024-01-01
1,France,Aurora,2024,Apr,2024-Apr,Central,Uncurtailed,Capture,Onshore wind,2023,...,68.806162,67.974667,68.182541,68.078604,69.117972,68.390414,NaN,NaN,4,2024-04-01
2,France,Aurora,2024,Jul,2024-Jul,Central,Uncurtailed,Capture,Onshore wind,2023,...,68.910098,67.974667,68.182541,67.974667,69.014035,68.390414,NaN,NaN,7,2024-07-01
3,France,Aurora,2024,Oct,2024-Oct,Central,Uncurtailed,Capture,Onshore wind,2023,...,68.598288,68.910098,69.014035,68.598288,68.702225,68.910098,NaN,NaN,10,2024-10-01
4,France,Aurora,2025,Jan,2025-Jan,Central,Uncurtailed,Capture,Onshore wind,2023,...,68.598288,68.910098,69.014035,68.598288,68.702225,68.910098,NaN,NaN,1,2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,Spain,Aurora,2025,Apr,2025-Apr,Central,NaN,Baseload,NaN,2024,...,66.8899,66.14384,66.82858,67.26804,67.51332,67.52354,NaN,NaN,4,2025-04-01
260,Spain,Aurora,2025,Jul,2025-Jul,Central,NaN,Baseload,NaN,2024,...,66.85924,66.13362,66.80814,67.27826,67.52354,67.53376,NaN,NaN,7,2025-07-01
261,Spain,Aurora,2025,Oct,2025-Oct,Central,NaN,Baseload,NaN,2024,...,68.64896,67.57376,68.32128,68.12672,68.4544,67.86048,NaN,NaN,10,2025-10-01
262,Spain,Aurora,2026,Jan,2026-Jan,Central,NaN,Baseload,NaN,2024,...,68.66944,67.55328,68.2496,68.03456,68.37248,67.76832,NaN,NaN,1,2026-01-01


In [7]:

# We drop columns that have at least one missing value, as we will later perform the variation
# of average price between consecutive forecasts. This way we ensure that the time periods we compare are the same across all forecasts.

# We now include a column with the average price across all technologies, to be used as a reference in the analysis.
# Take into account that year data are columns, so we need to specify axis=1 when calculating the mean.

# We first calclulate year period, based on columns

year_cols = df_raw_fil.columns[df_raw_fil.columns.str.contains(r'20\d{2}')]

# Coerce year columns to numeric — Excel separator strings like '---' become NaN
df_raw_fil[year_cols] = df_raw_fil[year_cols].apply(pd.to_numeric, errors='coerce')

df_final = df_raw_fil.copy()

df_final['Technology view'] = df_final.apply(lambda row: row['Scope'] if (pd.isna(row['Technology view']) and row['Scope'] == 'Baseload') else row['Technology view'], axis=1)

df_final['average_price'] = df_final[year_cols].mean(axis=1)
# We calculate the average of the variation: first we calculate 
dif_cols = [f'abs_variation {year}' for year in year_cols]
perc_dif_cols = [f'perc_variation {year}' for year in year_cols]

for dif_col, perc_dif_col, year_col in zip(dif_cols, perc_dif_cols, year_cols):
    df_final[dif_col] = df_final.groupby(['Country', 'Scenario', 'Scope', 'Technology view'])[year_col].diff()
    df_final[perc_dif_col] = df_final.groupby(['Country', 'Scenario', 'Scope', 'Technology view'])[year_col].pct_change(fill_method=None) * 100

df_final['variation of average price'] = df_final[dif_cols].mean(axis=1)
df_final['avg_price_variation_perc'] = df_final[perc_dif_cols].mean(axis=1)
# drop first row of each group, as it will have NaN value in the variation column
# df_final = df_final.dropna(subset=['variation of average price'], axis=1)

drop_list = ['Curtailed/uncurtailed', 'Year of the real currency', 'Transformed to Real year', 'Source', 'Scenario', 'Release year', 'Release Month/quarter', 'Month'] + year_cols.tolist() + dif_cols + perc_dif_cols
df_final.drop(columns=drop_list, inplace=True)
# df_final.dropna(subset=['variation of average price'], inplace=True)
df_final.to_excel('processed_prices.xlsx')

df_final

C:\Users\mpy\AppData\Local\Temp\ipykernel_21336\3518019214.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['variation of average price'] = df_final[dif_cols].mean(axis=1)
C:\Users\mpy\AppData\Local\Temp\ipykernel_21336\3518019214.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['avg_price_variation_perc'] = df_final[perc_dif_cols].mean(axis=1)


,Country,Release date,Scope,Technology view,Date,average_price,variation of average price,avg_price_variation_perc
0,France,2024-Jan,Capture,Onshore wind,2024-01-01,74.512615,NaN,NaN
1,France,2024-Apr,Capture,Onshore wind,2024-04-01,70.859616,-3.652999,-3.834313
2,France,2024-Jul,Capture,Onshore wind,2024-07-01,71.637737,0.778121,1.188292
3,France,2024-Oct,Capture,Onshore wind,2024-10-01,71.075917,-0.561821,-0.708886
4,France,2025-Jan,Capture,Onshore wind,2025-01-01,70.882010,-0.109711,-0.172579
...,...,...,...,...,...,...,...,...
259,Spain,2025-Apr,Baseload,Baseload,2025-04-01,73.906782,-0.174099,-0.247580
260,Spain,2025-Jul,Baseload,Baseload,2025-07-01,73.041204,-0.865577,-0.989765
261,Spain,2025-Oct,Baseload,Baseload,2025-10-01,73.537536,0.885308,1.242105
262,Spain,2026-Jan,Baseload,Baseload,2026-01-01,73.346194,-0.191342,-0.286816


In [8]:
dates_df = df_final[['Date']].dropna().drop_duplicates().sort_values(by='Date').reset_index(drop=True)
last_4_dates = dates_df['Date'].unique()[-4:]
last_4_dates

<DatetimeArray>
['2025-07-01 00:00:00', '2025-10-01 00:00:00', '2026-01-01 00:00:00',
 '2026-04-01 00:00:00']
Length: 4, dtype: datetime64[ns]

In [9]:

final_dict_deltas = {}
final_dict_abs = {}
dates_df = df_final[['Date']].dropna().drop_duplicates().sort_values(by='Date').reset_index(drop=True)
last_4_dates = dates_df['Date'].unique()[-4:]

# Get Release date labels in chronological order (Date is already sorted ascending)
ordered_release_dates = (
    df_final[df_final['Date'].isin(last_4_dates)]
    .sort_values(by='Date')
    [['Date', 'Release date']]
    .drop_duplicates(subset='Date')
    ['Release date']
    .tolist()
)

for technology in df_final['Technology view'].unique():
    df_tech = df_final[(df_final['Technology view'] == technology) & (df_final['Date'].isin(last_4_dates))].sort_values(by='Date', ascending=False).reset_index(drop=True)
    df_tech = df_tech[['Release date', 'Country', 'average_price', 'variation of average price']].reset_index(drop=True)

    df_abs_pivot = df_tech.pivot(index='Country', columns='Release date', values='average_price')[ordered_release_dates]
    final_dict_abs[technology] = df_abs_pivot

    df_deltas_pivot = df_tech.pivot(index='Country', columns='Release date', values='variation of average price')[ordered_release_dates]
    final_dict_deltas[technology] = df_deltas_pivot

final_dict_abs['Baseload'].columns.to_list()


['2025-Jul', '2025-Oct', '2026-Jan', '2026-Apr']

In [10]:
import json

# Assuming you have two dicts — one for series values, one for deltas
# series_dict = {"Onshore wind": df_series, "Solar PV": df_series_2, ...}
# deltas_dict = {"Onshore wind": df_deltas, "Solar PV": df_deltas_2, ...}

SERIES = {}
for technology, df in final_dict_abs.items():
    SERIES[technology] = {
        country: row.tolist()
        for country, row in df.iterrows()
    }

COUNTRIES = df_final['Country'].unique().tolist()

DELTAS = {}
for technology, df in final_dict_deltas.items():
    DELTAS[technology] = {
        country: row.tolist()
        for country, row in df.iterrows()
    }

with open('input.html', 'r') as f:
    template = f.read()

template = template.replace('{{SERIES}}', json.dumps(SERIES))
template = template.replace('{{DELTAS}}', json.dumps(DELTAS))
template = template.replace('{{COUNTRIES}}', json.dumps(COUNTRIES))

with open('output.html', 'w') as f:
    f.write(template)